# Section 5.4 — Scaling Limits of the SP Heuristic

Reproduces **Table 7** from the thesis.  
Runs the **SP heuristic (B-PHA)** on a synthetic 70-cage instance.

The 70-cage fleet uses the real Loc1-Loc6 data (62 cages) plus a synthetic Loc7
that mirrors Loc1 (10 cages) to reach 70 cages total. Loc7 MAB is set to 4 000 000 kg
and regional MAB is scaled proportionally (35 000 000 × 7/6).  
DE and EEV are not attempted — DE is out-of-memory at this scale.

In [ ]:
import sys
import os

_here = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(_here, '..', 'models'))
sys.path.insert(0, _here)  # local instance.py (70 cages) takes priority

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df,
    temps_bad_12, temps_normal_12, temps_good_12,
)
from SP import BinaryProgressiveHedging

In [ ]:
# 70-cage fleet: Loc1-Loc6 (real data) + Loc7 (synthetic, mirror of Loc1)

units_df_70 = units_df.copy().reset_index(drop=True)
loc_mab_70  = loc_mab

print(f'Fleet size: {len(units_df_70)} cages across {units_df_70["location"].nunique()} locations')
print(units_df_70.groupby('location').size().to_frame('cages'))
print(f'Regional MAB: {regional_mab:,.0f}')

In [ ]:
# SP heuristic on 70-cage fleet  (mip_gap=2%, matches thesis Table 7)

MIP_GAP = 0.02

print('Building SP model (70 cages)...')
ald = BinaryProgressiveHedging(
    units_df=units_df_70,
    loc_mab=loc_mab_70,
    regional_mab=regional_mab,
    T=T,
    mip_gap=MIP_GAP,
    temps_bad=temps_bad_12,
    temps_normal=temps_normal_12,
    temps_good=temps_good_12,
)
ald.build()
ald.solve()


In [ ]:
# Results — Table 7

print('\n' + '='*50)
print(f'Scenarios       : {ald.n_feasible}/{ald.n_scenarios} feasible')
print(f'E[obj] (MNOK)   : {ald.eval_obj / 1e6:.1f}')
print(f'MIP gap (%)     : {MIP_GAP * 100:.0f}')
print(f'Wall clock (s)  : {ald.total_time:.1f}')
print('='*50)

df_result = pd.DataFrame([{
    'Metric': 'Scenarios',
    'SP': f'{ald.n_feasible}/{ald.n_scenarios} feasible',
}, {
    'Metric': 'E[obj] (MNOK)',
    'SP': round(ald.eval_obj / 1e6, 1),
}, {
    'Metric': 'MIP gap tolerance (%)',
    'SP': MIP_GAP * 100,
}, {
    'Metric': 'Wall clock time (s)',
    'SP': round(ald.total_time, 1),
}]).set_index('Metric')

display(df_result)
